# pyologger workshop demo (NDP)

A self-contained walkthrough of the `pyologger` tag-data pipeline, built to run inside a fresh **National Data Platform (NDP)** workspace with no prior setup, no external drives, and no lab database access.

**What you'll do:**
1. Pull a small demo deployment from the lab's public [Pelican](https://docs.pelicanplatform.org/about-pelican)/OSDF namespace (`jkb-lab-public`) — no account or token needed.
2. Load it as a `DataReader` object, the same class the full pipeline uses.
3. Explore what's inside: signals, channels, sampling rates, and scored sleep-state events.
4. Build an interactive multi-signal dashboard — depth, EEG (with a live spectrogram), ECG, heart rate, stroke rate — with sleep states shaded on top.
5. Zoom into a single REM bout at full resolution.

**Example deployment:** `2020-04-10_mian-002` — a juvenile northern elephant seal (*Mirounga angustirostris*, nicknamed "SnoozySuzy") carrying EEG/EOG/EMG, ECG, and a motion + depth tag, recorded on land at Año Nuevo State Park. The full recording is 4.2 days; this notebook loads a pre-trimmed 3-hour demo slice (2020-04-12 09:00–12:00 local) so downloads and rendering stay fast.

> This notebook intentionally skips the metadata (Notion) and raw-import steps used internally by the lab — those need private credentials. Everything here comes from one public, read-only Pelican download.

## 0. One-time setup

If you're running this in a brand-new NDP workspace, install `pyologger` and its notebook dependencies first (uncomment and run once):

In [ ]:
# %pip install -e .. -q
# %pip install plotly plotly-resampler pandas xarray -q

## 1. Pull the demo deployment from Pelican

`pyologger` normally loads a lab deployment via `select_and_load_deployment()`, which expects a local folder tree described in `config.yaml`. Since this NDP workspace starts empty, this cell:

1. Writes a minimal `config.yaml` at the repo root, pointing `local_private_data` at a small `demo_data/` folder inside this repo (created below).
2. Downloads the demo deployment's pickled `DataReader` object (`data.pkl`) from the **public**, no-token-required `jkb-lab-public` Pelican namespace straight into that folder.

No Pelican token is needed for any step in this notebook — `jkb-lab-public` is openly readable.

In [ ]:
import os
import subprocess
import yaml

dataset_id = "mian-juv-nese_sleep_lml-ano_JKB"
deployment_id = "2020-04-10_mian-002"

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
demo_data_dir = os.path.join(repo_root, "demo_data")
deployment_outputs_dir = os.path.join(demo_data_dir, dataset_id, deployment_id, "outputs")
os.makedirs(deployment_outputs_dir, exist_ok=True)

config_path = os.path.join(repo_root, "config.yaml")
if not os.path.exists(config_path):
    minimal_config = {
        "paths": {
            "local_private_data": demo_data_dir,
            "local_public_data": demo_data_dir,
            "local_repo_path": repo_root,
        }
    }
    with open(config_path, "w") as f:
        yaml.safe_dump(minimal_config, f, sort_keys=False)
    print(f"✅ Wrote minimal config.yaml at {config_path}")
else:
    print(f"ℹ️ Using existing config.yaml at {config_path}")

print(f"Demo data folder: {demo_data_dir}")

In [ ]:
# Locate the pelican CLI, or fall back to `pip install pelicanfs` if it isn't on PATH.
pelican_bin = subprocess.run(["which", "pelican"], capture_output=True, text=True).stdout.strip()

pkl_remote_path = "osdf:///jkb-lab-public/demo/2020-04-10_mian-002_data.pkl"
pkl_local_path = os.path.join(deployment_outputs_dir, "data.pkl")

if pelican_bin:
    print(f"Using pelican CLI at {pelican_bin}")
    if not os.path.exists(pkl_local_path):
        result = subprocess.run(
            [pelican_bin, "object", "get", pkl_remote_path, pkl_local_path],
            capture_output=True, text=True,
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError("pelican object get failed — see stderr above.")
    print(f"✅ Downloaded to {pkl_local_path} ({os.path.getsize(pkl_local_path) / 1e6:.1f} MB)")
else:
    # Fallback: pelicanfs (fsspec), no CLI required.
    print("pelican CLI not found on PATH — falling back to pelicanfs (pip install pelicanfs).")
    import fsspec
    if not os.path.exists(pkl_local_path):
        with fsspec.open("osdf:///jkb-lab-public/demo/2020-04-10_mian-002_data.pkl", "rb") as src, \
             open(pkl_local_path, "wb") as dst:
            dst.write(src.read())
    print(f"✅ Downloaded to {pkl_local_path} ({os.path.getsize(pkl_local_path) / 1e6:.1f} MB)")

## 2. Load the deployment

This `.pkl` is a pickled `pyologger.load_data.datareader.DataReader` object — already imported, calibrated, and derived (zero-offset-corrected depth, oriented accelerometer/magnetometer, detected heartbeats/strokes, and scored sleep states). It's the exact same object type produced by the pipeline's `read_files()` step, just further downstream, so every plotting/analysis utility in `pyologger` works on it unchanged.

In [ ]:
import sys
sys.path.insert(0, repo_root)

import pickle
import pandas as pd

from pyologger.utils.folder_manager import load_configuration
from pyologger.utils.param_manager import ParamManager
from pyologger.plot_data.plotter import plot_tag_data_interactive, _expand_state_annotation_patterns, load_color_mapping

config, data_dir, color_mapping_path, montage_path = load_configuration()

with open(pkl_local_path, "rb") as f:
    data_pkl = pickle.load(f)

# The pkl was written with a stale output_folder from the original machine; repoint
# it at where we actually loaded it from.
data_pkl.output_folder = deployment_outputs_dir
data_pkl.deployment_folder = os.path.dirname(deployment_outputs_dir)

param_manager = ParamManager(deployment_folder=data_pkl.deployment_folder, deployment_id=deployment_id)

timezone = data_pkl.deployment_info["Time Zone"]
print(f"Deployment : {deployment_id}")
print(f"Animal     : {dataset_id}")
print(f"Time zone  : {timezone}")

## 3. What's inside?

Every signal lives in `data_pkl.signal_data` as a `pandas.DataFrame` with a `datetime` column, and its metadata (units, sampling rate, channel names) in `data_pkl.signal_info`.

In [ ]:
rows = []
for name, df in data_pkl.signal_data.items():
    if df is None or "datetime" not in df.columns or df.empty:
        continue
    rows.append({
        "signal": name,
        "samples": len(df),
        "channels": ", ".join(c for c in df.columns if c != "datetime"),
        "start": df["datetime"].iloc[0],
        "end": df["datetime"].iloc[-1],
    })

signal_overview = pd.DataFrame(rows).sort_values("samples", ascending=False).reset_index(drop=True)
signal_overview

Sleep states are stored in `data_pkl.event_data` as rows with `type == "state"`, each with a `datetime` (onset) and a `duration` in seconds. Two parallel scoring schemes are present: a detailed scheme (active/quiet waking, LV-SWS, HV-SWS, certain/putative REM) and a collapsed "simple" scheme (active waking, quiet waking, SWS, REM).

In [ ]:
events = data_pkl.event_data
state_events = events[events["type"] == "state"].copy()

summary = (
    state_events.groupby("key")
    .agg(n_events=("key", "size"), total_seconds=("duration", "sum"))
    .sort_values("total_seconds", ascending=False)
)
summary["total_minutes"] = (summary["total_seconds"] / 60).round(1)
summary[["n_events", "total_minutes"]]

## 4. Interactive dashboard

`plot_tag_data_interactive` returns a `plotly-resampler` figure: it renders a downsampled view and re-fetches full-resolution data as you zoom — what makes a multi-million-sample ECG trace usable in a notebook.

State annotations shade time spans (sleep states); note annotations mark individual points (detected heartbeats, strokes, dives). Colors and target signal rows both resolve automatically from `color_mappings.json`, which ships in this repo.

In [ ]:
state_annotations_detailed = {"sleep-state_*": {}}       # -> shaded on the EEG row
state_annotations_simple = {"simple_sleep_state.*": {}}  # -> shaded on the depth row
state_annotations = {**state_annotations_detailed, **state_annotations_simple}

notes_to_plot = {
    "heartbeat_auto_detect_accepted":  {"signal": "heart_rate",  "symbol": "triangle-up",   "color": "green"},
    "strokebeat_auto_detect_accepted": {"signal": "stroke_rate", "symbol": "triangle-up",   "color": "green"},
    "dive":                            {"signal": "depth",       "symbol": "triangle-down", "color": "blue"},
}

color_mapping = load_color_mapping(color_mapping_path)
available_keys = sorted(data_pkl.event_data["key"].astype(str).unique())
for key in _expand_state_annotation_patterns(state_annotations, available_keys):
    print(f"{key:42s} {color_mapping.get(key)}")

In [ ]:
TARGET_SAMPLING_RATE = 10  # Hz for the initial (zoomed-out) render

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "ecg", "heart_rate", "stroke_rate", "odba", "prh"],
    channels={
        "eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"],
        "prh": ["pitch", "roll"],
    },
    state_annotations=state_annotations,
    note_annotations=notes_to_plot,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_range_selector_channel="depth",
    spectrogram_channel="eeg_p4",   # STFT computed from this channel, drawn above its row
    spectrogram_range=(0, 10),      # delta + theta band — sleep EEG is mostly under 10 Hz
    spectrogram_contrast=(2, 98),   # dB percentiles used as color limits
)

fig

## 5. Zoom into a single REM bout

Pass `zoom_start_time` / `zoom_end_time` to open the figure already zoomed into a window, while keeping the surrounding context available when you zoom back out.

In [ ]:
rem_events = state_events[state_events["key"].str.contains("rem", case=False, na=False)]

if not rem_events.empty:
    bout = rem_events.loc[rem_events["duration"].idxmax()]
    bout_start = bout["datetime"]
    bout_end = bout_start + pd.Timedelta(seconds=float(bout["duration"]))
    print(f"Longest REM bout: {bout['key']}")
    print(f"  {bout_start} -> {bout_end}  ({bout['duration']:.0f} s)")

    pad = pd.Timedelta(minutes=5)
    ZOOM_START, ZOOM_END = bout_start - pad, bout_end + pad
else:
    print("No REM events in this window; falling back to the first 30 minutes.")
    ZOOM_START = data_pkl.signal_data["depth"]["datetime"].iloc[0]
    ZOOM_END = ZOOM_START + pd.Timedelta(minutes=30)

In [ ]:
fig_bout = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "eog", "ecg", "heart_rate"],
    channels={
        "eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"],
        "eog": ["eog_l", "eog_r"],
    },
    state_annotations=state_annotations_detailed,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=100,   # higher rate for a short span
    zoom_start_time=ZOOM_START,
    zoom_end_time=ZOOM_END,
    zoom_range_selector_channel="depth",
    state_annotation_channel_mode="combined",
    state_annotation_channel_height_ratio=0.15,
)

fig_bout

## Where to go from here

- Drag your own tag data into this workspace's file browser and point `dataset_folder` / `deployment_id` at it, then walk through `00_load_data.ipynb` → `01_calibrate_pressure.ipynb` → `02_calibrate_accmag.ipynb` for the full raw-to-processed pipeline this demo skipped.
- Everything you downloaded lives under `demo_data/` in this repo — safe to delete and re-run this notebook from scratch at any time.
- The lab's private namespace (`jkb-lab`) works the same way but needs a bearer token — see the Pelican Wiki if you want to publish or pull your own protected data.